# Analyse toevoegen handleiding instructies
Dit notebook ondersteunt het verkennen, analyseren en valideren van de handleiding en instructies functionaliteit binnen AskDelphi.  
De module biedt helpermethoden voor het toevoegen, beheren en inspecteren van handleiding en instructie relaties die aan AskDelphi‑topics gekoppeld zijn.  
Het doel van dit notebook is om:

- inzicht te krijgen in de datastructuren en API‑interacties rondom handleidingen en instructies
- voorbeelddata te analyseren en patronen te ontdekken
- helpermethoden te testen en gedrag te evalueren
- experimentele of onderzoeksgerichte analyses uit te voeren ter verbetering van de contentstructuur in AskDelphi

Dit notebook vormt daarmee een flexibel startpunt voor verdere exploratie of debugging van de Relations & Tags endpoint.

### Connectie met AskDelphi opzetten

In [2]:
import uuid
from ask_delphi_api.authentication import AskDelphiClient
client = AskDelphiClient()
client.authenticate()   # pakt automatisch portal code uit .env

ModuleNotFoundError: No module named 'ask_delphi_api'

In [ ]:
from ask_delphi_api.project import Project
from ask_delphi_api.topictools import TopicTools
from ask_delphi_api.relation import Relation
from ask_delphi_api.workflow import Workflow
workflow = Workflow(client)
project = Project(client)
topic = TopicTools(client, project)
relation = Relation(client)

### Creatie relatie naar handleiding

In [3]:
# Create Action topic
topic_id_action = topic.topic_upload("Instructie handleiding", "Action")
result = topic.checkout(topic_id_action)
topic_version_id_action = result['topicVersionId']
print(f"topic_id_action : {topic_id_action}")
print(f"topic_version_id_action : {topic_version_id_action}")

topic_id_action : 7e4b5291-f516-46aa-b446-40f14beb97e3
topic_version_id_action : e2a0ca1f-8f98-42ba-8ac4-05e973b73ae2


In [4]:
# Welke relatie types zijn er?
endpoint = f"/v2/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topic_id_action}/topicVersion/{topic_version_id_action}/allowedrelations"
result = client._request(
    "GET", 
    endpoint,
    json_data={}
)

# relationTypeId verschilt per topicType
for item in result["topicAllowedRelations"]:
    if item['pyramidLevelName'] == "Handleidingen en Instructies":
        print(item['topicTypeName'], item["relationTypeId"])
        # print(item)

Bestand 29c925e3-e0c3-4697-a7aa-01b4258b46b1
Ondersteunende kennis 29c925e3-e0c3-4697-a7aa-01b4258b46b1
Externe link 29c925e3-e0c3-4697-a7aa-01b4258b46b1
Video 29c925e3-e0c3-4697-a7aa-01b4258b46b1
Inhoud / Artikel 29c925e3-e0c3-4697-a7aa-01b4258b46b1
ConnectPeople d35774e1-eaba-4cc6-89d1-e7c7eae50d56
SharePoint d35774e1-eaba-4cc6-89d1-e7c7eae50d56
Intranet d35774e1-eaba-4cc6-89d1-e7c7eae50d56


In [5]:
# wat is relation_type_id ConnectPeople?
relation_type_id = relation.get_relation_type_id(topic_id_action, topic_version_id_action, "ConnectPeople")
print(f"relation_type_id ConnectPeople : {relation_type_id}")

relation_type_id ConnectPeople : 614377d0-0e36-4f7d-abb2-d62ad1eb305d


In [6]:
# wat is relation_type_id Bestand?
relation_type_id = relation.get_relation_type_id(topic_id_action, topic_version_id_action, "Bestand")
print(f"relation_type_id Bestand : {relation_type_id}")

relation_type_id Bestand : 29c925e3-e0c3-4697-a7aa-01b4258b46b1


In [7]:
# wat is het topic type van ConnectPeople?
topic_type_id_connectpeople = project.get_topic_type_id("ConnectPeople")
print(f"topic_type_id: {topic_type_id_connectpeople}")

topic_type_id: e5a7b60c-5b45-45b5-8c00-2bfaa4b068b4


In [8]:
# wat is het topic type van Questionnaire?
topic_type_id_questionnaire = project.get_topic_type_id("Questionnaire")
print(f"topic_type_id: {topic_type_id_questionnaire}")

topic_type_id: c1225506-63e2-4785-9e51-06a587d54a9c


In [9]:
# Toevoegen ConnectPeople topic aan action
topic_id_connectpeople = str(uuid.uuid4())      
topicTitle = "Handleiding ConnectPeople"      
topicTypeId = project.get_topic_type_id("ConnectPeople")    
parentTopicId = topic_id_action
parentTopicRelationTypeId = relation.get_relation_type_id(topic_id_action, topic_version_id_action, "ConnectPeople")
parentTopicVersionId = topic_version_id_action
relation.add_topic_with_relation(client, topic_id_connectpeople, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)

{'topicId': '714f3b04-8420-43dc-a1ce-3af13d0806d1',
 'topicVersionKey': '770beffa-3414-44da-8927-0f4a8d20144f'}

In [10]:
# Toevoegen Questionnaire topic aan action
topic_id_questionnaire = str(uuid.uuid4())      
topicTitle = "Handleiding questionnaire"      
topicTypeId = project.get_topic_type_id("Questionnaire")    
parentTopicId = topic_id_action
parentTopicRelationTypeId = relation.get_relation_type_id(topic_id_action, topic_version_id_action, "Bestand")
parentTopicVersionId = topic_version_id_action
relation.add_topic_with_relation(client, topic_id_questionnaire, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)

{'topicId': 'c6bd21c4-595c-4ba1-b8b8-f6744214401d',
 'topicVersionKey': 'e00ee4e5-626d-4f3d-8b4e-5aa5cb529e83'}

In [12]:
# Checkin taak topic
topic.checkin(topic_id_action)
print(f"Checked in task topic : {topic_id_action}")

Checked in task topic : 7e4b5291-f516-46aa-b446-40f14beb97e3


In [14]:
# Toevoegen handleiding aan bestaande digicoach
topic_id_action = "2fa40104-795a-47e6-bd5f-a0b428ab14c7"
result = topic.checkout(topic_id_action)
topic_version_id_action = result['topicVersionId']
print(f"topic_id_action : {topic_id_action}")
print(f"topic_version_id_action : {topic_version_id_action}")

# Toevoegen Questionnaire topic aan action
topic_id_questionnaire = str(uuid.uuid4())      
topicTitle = "Handleiding questionnaire"      
topicTypeId = project.get_topic_type_id("Questionnaire")    
parentTopicId = topic_id_action
parentTopicRelationTypeId = relation.get_relation_type_id(topic_id_action, topic_version_id_action, "Bestand")
parentTopicVersionId = topic_version_id_action
relation.add_topic_with_relation(client, topic_id_questionnaire, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)

# Checkin taak topic
topic.checkin(topic_id_action)
print(f"Checked in task topic : {topic_id_action}")

topic_id_action : 2fa40104-795a-47e6-bd5f-a0b428ab14c7
topic_version_id_action : 2cad5c89-4475-492b-8f0e-b53b3dda856b
Checked in task topic : 2fa40104-795a-47e6-bd5f-a0b428ab14c7


In [11]:
# # document_part toevoegen aan Instructie handleiding
# endpoint = f"/v3/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topic_id_bestand}/topicVersion/{topic_version_id_bestand}/part"
# document_part = {
#     "name": "documenten",
#     "content": {
#         "documents": [
#             {
#                 "title": "Aangifte Schenkbelasting 2025",
#                 "url": "https://download.belastingdienst.nl/belastingdienst/docs/aangifte_schenkbelasting_2025_suc0632z52fol.pdf",
#                 "type": "PDF",
#                 "description": "formulier"
#             }
#         ]
#     }
# }
# result = client._request(
#     "PUT", 
#     endpoint,
#     json_data=document_part
# )
# print(result)

In [15]:
import uuid

# Constant
DIGICOACH_NAME = "Digicoach test arie 11feb"
TASK_NAME = "Taak test 11feb"
ACTION_NAME = "Stap test 10feb"
PREDEFINED_SEARCH_NAME = "Voorgedefinieerde zoekopdracht 11feb"

# Create Voorgedefinieerde zoekopdracht topic
topic_id_predefined_search = topic.topic_upload(PREDEFINED_SEARCH_NAME, "Pre-defined search")
topic_version_id_predefined_search = topic.get_topicVersionId(topic_id_predefined_search)
print(f"Created Voorgedefinieerde zoekopdracht topic : {topic_id_predefined_search}")

# Create Digicoach topic
topic_id_digicoach = str(uuid.uuid4())      
topicTitle = DIGICOACH_NAME      
topicTypeId = project.get_topic_type_id("Digitale Coach Procespagina")     
parentTopicId = topic_id_predefined_search
parentTopicRelationTypeId = relation.get_relation_type_id(topic_id_predefined_search, topic_version_id_predefined_search,"Voorgedefinieerde zoekopdracht")
parentTopicVersionId = topic_version_id_predefined_search
relation.add_topic_with_relation(client, topic_id_digicoach, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)
print(f"Created Digicoach topic : {topic_id_digicoach}")

# Get Digicoach topic version ID
topic_version_id_digicoach = topic.get_topicVersionId(topic_id_digicoach)

# Tag Digitale Coach Procespagina
relation.add_tag(topic_id_digicoach, topic_version_id_digicoach, "interactie")

# Create Task topic
topic_id_task = str(uuid.uuid4())
topicTitle = TASK_NAME       
topicTypeId = project.get_topic_type_id("Task")     
parentTopicId = topic_id_digicoach
parentTopicRelationTypeId = relation.get_relation_type_id(topic_id_digicoach, topic_version_id_digicoach, "Taak")
parentTopicVersionId = topic_version_id_digicoach
relation.add_topic_with_relation(client, topic_id_task, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)
print(f"Created Task topic : {topic_id_task}")

# Get Task topic version ID
topic_version_id_task = topic.get_topicVersionId(topic_id_task)

# Create Action topic
topic_id_action = str(uuid.uuid4())
topicTitle = ACTION_NAME       
topicTypeId = project.get_topic_type_id("Action")     
parentTopicId = topic_id_task
parentTopicRelationTypeId = relation.get_relation_type_id(topic_id_task, topic_version_id_task, "Stap")
parentTopicVersionId = topic_version_id_task
relation.add_topic_with_relation(client, topic_id_action, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)
print(f"Created Action topic : {topic_id_action}")

# Get Action topic version ID
topic_version_id_action = topic.get_topicVersionId(topic_id_action)

# Toevoegen Questionnaire topic aan action
topic_id_questionnaire = str(uuid.uuid4())      
topicTitle = "Handleiding questionnaire"      
topicTypeId = project.get_topic_type_id("Questionnaire")    
parentTopicId = topic_id_action
parentTopicRelationTypeId = relation.get_relation_type_id(topic_id_action, topic_version_id_action, "Bestand")
parentTopicVersionId = topic_version_id_action
relation.add_topic_with_relation(client, topic_id_questionnaire, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)

# Checkin Voorgedefinieerde zoekopdracht topic
topic.checkin(topic_id_predefined_search)
print(f"Checked in Voorgedefinieerde zoekopdracht topic : {topic_id_predefined_search}")

# Checkin Digicoach topic
topic.checkin(topic_id_digicoach)
print(f"Checked in Digicoach topic : {topic_id_digicoach}")

# Checkin Digicoach taak topic
topic.checkin(topic_id_task)
print(f"Checked in Digicoach task topic : {topic_id_task}")

# Checkin Digicoach stap topic
topic.checkin(topic_id_action)
print(f"Checked in Digicoach action topic : {topic_id_action}")

# # Delete Task topic
# topic_version_id_task = topic.get_topicVersionId(topic_id_task)
# topic.delete_topic(topic_id_task, topic_version_id_task)
# print(f"Deleted Task topic : {topic_id_task}")

# # Delete Digicoach topic
# topic.delete_topic(topic_id_digicoach, topic_version_id_digicoach)
# print(f"Deleted Digicoach topic : {topic_id_digicoach}")

# # Delete Voorgedefinieerde zoekopdracht topic
# topic.delete_topic(topic_id_predefined_search, topic_version_id_predefined_search)
# print(f"Deleted Voorgedefinieerde zoekopdracht topic : {topic_id_predefined_search}")

Created Voorgedefinieerde zoekopdracht topic : 3b5c4d9f-c0c9-4f2e-b6c8-718b38b2b48b
Created Digicoach topic : 7cc0420c-a2d7-4ec8-b62f-9e7dbf182c3d
Created Task topic : f7df45a3-33ab-457c-8433-349fd7209c81
Created Action topic : 06fc5029-fd84-4e02-956d-a37a2c7e0b32
Checked in Voorgedefinieerde zoekopdracht topic : 3b5c4d9f-c0c9-4f2e-b6c8-718b38b2b48b
Checked in Digicoach topic : 7cc0420c-a2d7-4ec8-b62f-9e7dbf182c3d
Checked in Digicoach task topic : f7df45a3-33ab-457c-8433-349fd7209c81
Checked in Digicoach action topic : 06fc5029-fd84-4e02-956d-a37a2c7e0b32


In [16]:
#  Creates a workflow transition request for predefined_search topic.
request_id = workflow.create_workflow_transition_request(topic_id_predefined_search)
transitions_model = workflow.get_workflow_transition_request_transitions_model(request_id)
workflow.update_workflow_transition_request(request_id, transitions_model)
workflow.approve_workflow_transition_request(request_id)

#  Creates a workflow transition request for digicoach topic.
request_id = workflow.create_workflow_transition_request(topic_id_digicoach)
transitions_model = workflow.get_workflow_transition_request_transitions_model(request_id)
workflow.update_workflow_transition_request(request_id, transitions_model)
workflow.approve_workflow_transition_request(request_id)

#  Creates a workflow transition request for task topic.
request_id = workflow.create_workflow_transition_request(topic_id_task)
transitions_model = workflow.get_workflow_transition_request_transitions_model(request_id)
workflow.update_workflow_transition_request(request_id, transitions_model)
workflow.approve_workflow_transition_request(request_id)

2026-02-11T16:55:23Z
2026-02-11T16:55:25Z
2026-02-11T16:55:26Z


{}